# DICOM → BIDS with HeuDiConv

This notebook converts raw MRI DICOMs into a [BIDS](https://bids.neuroimaging.io/) dataset using [HeuDiConv](https://heudiconv.readthedocs.io/).

**The workflow (two passes):**

1. **Discovery pass** – run HeuDiConv with the `convertall` heuristic and `-c none`. Nothing is converted; it just writes `dicominfo.tsv`, a table of every series in your scan.
2. **Look at the table, write a heuristic** – a small Python file that says "series named *mprage* → `anat/..._T1w`", etc.
3. **Conversion pass** – run HeuDiConv again with your heuristic to produce NIfTI + JSON sidecars + BIDS files.
4. **QC + deface** – inspect the output, deface the T1w/T2w with pydeface (keeping an unfaced backup outside the dataset), and check the before/after images.
5. **Copy** the `sub-*` files into your main BIDS dataset and run the BIDS validator there.

> 🖥️ **Where to run this:** run the notebook *on the server*, since the DICOMs and the Singularity image live there.
>
> 📦 **Get the singularity images once (pin a version so your conversions are reproducible):**
>
> If you don't have these containers, please create folders for them and install with these commands.
>
> Create a hidden cache directory in your own partition, rather than the default home folder location, plus a folder for each container:
>
> ```bash
> export APPTAINER_CACHEDIR=/jukebox/[yourpath]/singularity-containers/cache
> mkdir -p $APPTAINER_CACHEDIR
> mkdir -p /jukebox/[yourpath]/singularity-containers/{heudiconv,pydeface,bids-validator}
> ```

>
> Pull the following containers if you don't already have them:
> 
> ```bash
> apptainer pull /jukebox/[yourpath]/singularity-containers/heudiconv/heudiconv-1.4.0.sif \
>     docker://nipy/heudiconv:1.4.0
>
> apptainer pull /jukebox/[yourpath]/singularity-containers/pydeface/pydeface-2026-08-10.sif \
>     docker://poldracklab/pydeface@sha256:c648b8d70f31a186ff003ed2ad15d491d7b4a7d9adaa4665cf95526415fe3773
> ```
>

> For BIDS Validator, if you already installed it via pip, you can uninstall it first, and then pull the container:
>
> ```bash
> python -m pip uninstall bids-validator-deno
> apptainer pull /jukebox/[yourpath]/singularity-containers/bids-validator/bids-validator-3.0.2.sif \
>     docker://bids/validator:3.0.2
> ```

> ⚠️ **PHI warning:** DICOM headers contain identifying info. HeuDiConv keeps copies of headers in `<output>/.heudiconv/`. Don't push that folder to GitHub / share it without checking.

### Install the Python packages this notebook uses

This notebook runs in the same `pygers-meg` environment as the earlier notebooks, which already has `pandas` and `matplotlib`. It needs two more packages:

* **pydicom** reads DICOM headers (Step 2)
* **nibabel** loads the NIfTI images for the defacing check (Step 7)

Activate your environment and install them:

```bash
conda activate pygers-meg
pip install pydicom nibabel
```

Check they installed (this should print `ok`):

```bash
python -c "import pydicom, nibabel; print('ok')"
```

If Jupyter was already running, restart the kernel (**Kernel → Restart Kernel**) so it picks up the new packages.

## 0. Configure

Edit the cell below, then run everything top to bottom. Only this cell and the heuristic rules in Step 4 should need changes.

**`DICOM_TEMPLATE`** tells HeuDiConv how your DICOM folders are laid out, relative to `DICOM_ROOT`. Use `{subject}` (and `{session}` if you have sessions) as placeholders. Examples:

| Your folder layout | `DICOM_TEMPLATE` |
|---|---|
| `raw/sub01/SeriesA/img001.dcm` | `{subject}/*/*` |
| `raw/sub01/ses1/SeriesA/img001.dcm` | `{subject}/{session}/*/*` |
| `raw/sub01/*.dcm` (flat) | `{subject}/*` |

In [ ]:
from pathlib import Path

# ---- EDIT THESE LINES -------------------------------------------------------

# ---- Paths -----------------------------------------------------------------
DICOM_ROOT = Path("/Volumes/pinsk/MEG/pygers-meg/data/auditory-oddball")      # folder containing one folder per subject
BIDS_OUT   = Path("/Volumes/pinsk/MEG/pygers-meg/data/bids_staging")    # STAGING folder for the conversion (created if missing)
# Path to your larger BIDS dataset (the one with /meg). Only used by the optional copy step; leave None to skip.
MAIN_DATASET = Path("/Volumes/pinsk/MEG/pygers-meg/data/bids")

# Where the heuristic file (the rules that map your scans to BIDS names) will be saved.
# Any folder you can write to works; it is created for you in Step 4 if it doesn't exist.
# Keeping it with your study's other code makes it easy to find and reuse later.
# Tip: if you already have a main BIDS dataset, a good choice is its code/ folder,
# which BIDS allows. The heuristic then lives with the dataset it built, so anyone can
# see exactly how the MRI files were named (good for reproducibility):
#     HEURISTIC_PATH = MAIN_DATASET / "code" / "heuristic.py"
HEURISTIC_PATH = Path("/Volumes/pinsk/MEG/pygers-meg/code/heuristic.py")

# ---- Paths to singularity images -----------------------------------------------------------------
CONTAINER_DIR = Path("/jukebox/pinsk/singularity-containers")   # same folder you pulled into (see top of notebook)
SIF_PATH      = CONTAINER_DIR / "heudiconv" / "heudiconv-1.4.0.sif"
PYDEFACE_SIF  = CONTAINER_DIR / "pydeface" / "pydeface-2026-08-10.sif"
VALIDATOR_SIF = CONTAINER_DIR / "bids-validator" / "bids-validator-3.0.2.sif"

# ---- Layout of your DICOMs (relative to DICOM_ROOT) -------------------------
DICOM_TEMPLATE = "sub-{subject}/raw/mri/dcm/*.dcm"

# ---- Which subject(s) / session to convert ----------------------------------
# Subject label must match the folder name in DICOM_ROOT; BIDS output labels
# can only contain letters and numbers (no underscores/dashes).
SUBJECTS = ["001"]
SESSION  = "01"        # -> ses-01. Use None if there are no sessions


# ---- CHECK THE PATHS ABOVE (nothing below here needs editing) ---------------
# This prints a ✅ / ⚠️ / ❌ line for each folder and container image, so you can
# catch typos before running anything heavy.
#   ✅ = found
#   ⚠️ = not found yet, but that's OK (the notebook will create it, or the step is optional)
#   ❌ = missing and needed, so fix the path (or pull the image) before going on

problems = []   # we collect every ❌ here and report them all at the end

def check(label, path, kind="folder", required=True, created_later=False):
    """Print one status line for `path` and remember it if it's a real problem."""
    if path is None:                                      # e.g. MAIN_DATASET = None
        print(f"⚠️  {label:<14} not set (optional step will be skipped)")
        return
    path = Path(path)
    if "[yourpath]" in str(path):                         # placeholder never filled in
        print(f"❌  {label:<14} still contains [yourpath] -> {path}")
        problems.append(label)
        return
    found = path.is_file() if kind == "file" else path.is_dir()
    if found:
        print(f"✅  {label:<14} {path}")
    elif created_later and path.parent.is_dir():          # e.g. BIDS_OUT: made automatically later
        print(f"⚠️  {label:<14} doesn't exist yet; it will be created -> {path}")
    elif not required:
        print(f"⚠️  {label:<14} not found (only needed for an optional step) -> {path}")
    else:
        print(f"❌  {label:<14} NOT FOUND -> {path}")
        problems.append(label)

print("Data folders:")
check("DICOM_ROOT",    DICOM_ROOT)
check("BIDS_OUT",      BIDS_OUT, created_later=True)
check("MAIN_DATASET",  MAIN_DATASET, required=False)
if "[yourpath]" in str(HEURISTIC_PATH):                   # placeholder never filled in
    print(f"❌  {'HEURISTIC_PATH':<14} still contains [yourpath] -> {HEURISTIC_PATH}")
    problems.append("HEURISTIC_PATH")
elif HEURISTIC_PATH.exists():
    print(f"✅  {'HEURISTIC_PATH':<14} {HEURISTIC_PATH}  (will be overwritten in Step 4)")
else:
    print(f"⚠️  {'HEURISTIC_PATH':<14} doesn't exist yet; Step 4 will write it -> {HEURISTIC_PATH}")

print("\nContainer images:")
check("CONTAINER_DIR", CONTAINER_DIR)
check("SIF_PATH",      SIF_PATH,      kind="file")
check("PYDEFACE_SIF",  PYDEFACE_SIF,  kind="file")
check("VALIDATOR_SIF", VALIDATOR_SIF, kind="file")

# Stop here if anything is missing, so later cells don't fail with confusing errors.
if problems:
    raise FileNotFoundError("Fix these before continuing: " + ", ".join(problems) +
                            "  (missing images? see the pull commands at the top of the notebook)")
print("\n🎉 All paths look good. Carry on!")


In [ ]:
# ---- HeuDiConv settings -----------------------------------------------------
# Options that control how HeuDiConv converts and names your files.
# The defaults below work for most studies, so you can usually leave them as they are.
EXTRA_BINDS = []                 # extra folders to mount, e.g. ["/scratch", "/data/shared"]
OVERWRITE = True                # True -> redo conversions that already exist

# Add _run-01, _run-02... to every filename? Leave False if each scan is acquired once
# per session (cleaner names). Set True if a scan is ever repeated, so repeats get numbered.
ADD_RUN = False

# These scans will join a larger MEG BIDS dataset, so do NOT create top-level files
# (dataset_description.json, participants.tsv/json, README, CHANGES). True -> heudiconv "-b notop".
SKIP_TOP_LEVEL = True


## 1. Check your environment

In [ ]:
# ---- Tools used by the rest of the notebook ---------------------------------
import os          # file operations (e.g. swapping in the defaced image in Step 7)
import datetime    # timestamps for the defacing log
import shutil      # find programs on this computer, copy files
import subprocess  # run terminal commands from inside Python


def sh(cmd, check=False):
    """Run a terminal command and show its output live, as it happens.

    cmd   : the command as a list of words, e.g. ["ls", "-l", "/some/folder"]
    check : if True, stop the notebook with an error when the command fails

    Returns the command's exit code: 0 means it worked, anything else means it failed.
    Later cells use it whenever they run a program, e.g. sh(heudiconv_cmd(...)) in Steps 3 and 5,
    pydeface in Step 7, rsync in Step 8 and the BIDS validator in Step 9.
    """
    cmd = [str(word) for word in cmd]          # Path objects -> plain text
    print("$", " ".join(cmd), "\n")            # show the exact command, so you could copy it into a terminal

    # Start the command. stderr (error messages) is merged into stdout so you see everything in order.
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:                   # print each line as soon as the command writes it
        print(line, end="")

    rc = proc.wait()                           # wait for the command to finish; rc = its exit code
    if check and rc != 0:
        raise RuntimeError(f"Command failed with exit code {rc}")
    return rc


# ---- Find Apptainer (the program that runs the .sif containers) ---------------
# shutil.which() returns the full path to a program, or None if it isn't installed.
# Newer clusters call it "apptainer", older ones "singularity"; they work the same.
SING = shutil.which("apptainer") or shutil.which("singularity")

if SING is None:
    raise RuntimeError("Apptainer/Singularity not found. Are you running this notebook on the cluster? "
                       "(It won't work on a laptop.)")
print("✅ Apptainer found:", SING, "\n")

# Ask the HeuDiConv container for its version. This proves the container actually runs,
# and leaves a record in the notebook of which version did your conversion.
sh([SING, "run", "--cleanenv", SIF_PATH, "--version"])


## 2. Peek at your DICOMs

A quick sanity check that the paths/template are right *before* running anything heavy.

In [ ]:
import glob                    # find files that match a pattern like "*.dcm"
from collections import Counter  # count how many files belong to each series

for sub in SUBJECTS:
    # Fill the subject (and session) into the template, then list every matching file
    pattern = DICOM_TEMPLATE.format(subject=sub, session=SESSION or "*")
    files = sorted(glob.glob(str(DICOM_ROOT / pattern)))
    print(f"sub-{sub}: {len(files)} files match  {pattern}")
    if not files:
        print("   -> nothing matched. Check DICOM_ROOT, SUBJECTS and DICOM_TEMPLATE.")
        continue

    try:
        import pydicom
    except ImportError:
        print("   (pydicom isn't installed, so the headers can't be shown. See 'Install the Python packages' at the top.)")
        continue

    # ---- Study-level info: the same for every file, so read it from the first one ----
    ds = pydicom.dcmread(files[0], stop_before_pixels=True)   # header only, skip the image data (fast)
    for tag in ["PatientID", "StudyDate", "Manufacturer", "MagneticFieldStrength"]:
        print(f"   {tag}: {ds.get(tag, 'n/a')}")

    # ---- Series-level info: read every file's header and count files per series ----
    series = Counter()
    for f in files:
        try:
            h = pydicom.dcmread(f, stop_before_pixels=True,
                                specific_tags=["SeriesNumber", "SeriesDescription"])  # only read what we need
        except Exception:
            continue                                   # skip anything that isn't a readable DICOM
        series[(int(h.get("SeriesNumber", 0)), str(h.get("SeriesDescription", "n/a")))] += 1

    # Print one line per series, in the order they were acquired
    print(f"\n   {len(series)} series found:")
    print(f"   {'#':>4}  {'SeriesDescription':<45} files")
    for (num, desc), n in sorted(series.items()):
        print(f"   {num:>4}  {desc:<45} {n:>5}")


### 🔒 Look at the `PatientID` line

DICOM headers often hold **identifying information** (PHI): names, initials, birth dates, scan dates. In the sample dataset, `PatientID` contains the participant's **initials**. That shouldn't happen: when registering a participant at the scanner, enter a **study code** (e.g. `sub-001`) as the Patient ID, never their name or initials.

What this means as you go through the notebook:

* **The output above now contains PHI.** Do not share this notebook unles you first clear the outputs (**Kernel → Restart Kernel and Clear All Outputs**).
* **The BIDS files are safer.** dcm2niix (used by HeuDiConv) leaves identifying fields such as `PatientID` out of the JSON sidecars by default.
* **HeuDiConv keeps copies of the raw headers** in the hidden `.heudiconv/` folder inside `BIDS_OUT`. Never share or copy that folder; Step 8 deliberately leaves it behind.

## 3. Discovery pass → `dicominfo.tsv`

Before HeuDiConv can convert anything, *you* have to tell it which scan is which (that's the heuristic in Step 4). To write those rules, you first need to know what's in the scan session. That's what this step is for.

We run HeuDiConv with two special settings:

* **`-f convertall`**, a built-in heuristic that simply lists every series it finds
* **`-c none`**, which means "don't convert anything"

So **nothing is converted** here. HeuDiConv reads every DICOM header, groups the files into series, and writes a table describing them: `dicominfo.tsv`. It's saved in the hidden folder `BIDS_OUT/.heudiconv/<subject>/ses-<session>/info/`.

The cell below builds the HeuDiConv command and runs it (it takes a minute or so).

> 💡 The full command is printed before it runs (the line starting with `$`). You can copy it into a terminal to run HeuDiConv by hand.

In [ ]:
def heudiconv_cmd(subject, heuristic, converter, bids=True):
    """Build the full Apptainer command that runs HeuDiConv for one subject.

    subject   : subject label, e.g. "001"
    heuristic : "convertall" for this discovery pass, or the path to your heuristic file (Step 5)
    converter : "none" = only catalog the series (this step); "dcm2niix" = really convert (Step 5)
    bids      : True -> write BIDS-style output (only matters when actually converting)

    This function is defined once here and reused in Step 5 for the real conversion.
    It only BUILDS the command (a list of words); sh() is what actually runs it.
    """
    BIDS_OUT.mkdir(parents=True, exist_ok=True)       # make the staging folder if it doesn't exist yet

    # ---- HeuDiConv's own options ----------------------------------------------
    args = ["-s", subject,          # -s : which subject to process
            "-f", heuristic,        # -f : which heuristic (naming rules) to use
            "-c", converter]        # -c : which converter to use ("none" = don't convert)
    if bids and converter != "none":
        # -b        : write BIDS files (sidecars, scans.tsv, etc.)
        # -b notop  : same, but skip top-level files like dataset_description.json (see SKIP_TOP_LEVEL)
        args += ["-b", "notop"] if SKIP_TOP_LEVEL else ["-b"]
    if SESSION:
        args += ["-ss", SESSION]    # -ss : session label -> ses-01 in the output names
    if OVERWRITE:
        args += ["--overwrite"]     # redo a subject even if it was converted before

    # ---- Folders the container is allowed to see ("mounts" / "binds") -----------
    # A container can't see your files unless you "bind" a real folder to a path inside it.
    # Each entry is (folder on the cluster, path inside the container, read-only "ro" or read-write "rw").
    mounts = [(DICOM_ROOT.resolve(), "/data", "ro"),   # DICOMs: read-only, so they can't be changed by accident
              (BIDS_OUT.resolve(),   "/base", "rw")]   # staging folder: HeuDiConv writes its output here
    if heuristic != "convertall":
        # A custom heuristic file lives outside the container, so bind its folder at /code
        # and point -f at the file's path INSIDE the container.
        HEURISTIC_PATH.parent.mkdir(parents=True, exist_ok=True)
        mounts.append((HEURISTIC_PATH.parent.resolve(), "/code", "ro"))
        args[args.index("-f") + 1] = f"/code/{HEURISTIC_PATH.name}"
    for extra in EXTRA_BINDS:                          # any extra folders from the HeuDiConv settings cell
        mounts.append((Path(extra).resolve(), str(extra), "rw"))

    # -d : where the DICOMs are (as seen inside the container); HeuDiConv fills in {subject}/{session}
    # -o : where to write the output (as seen inside the container)
    tool_args = ["-d", f"/data/{DICOM_TEMPLATE}", "-o", "/base"] + args

    # Turn the mounts into "-B cluster_folder:container_path:mode" pairs
    binds = [x for src, dst, mode in mounts for x in ("-B", f"{src}:{dst}:{mode}")]

    # Final command: apptainer run --cleanenv -B ... -B ... heudiconv.sif <heudiconv options>
    # --cleanenv keeps your own environment settings from leaking into the container
    return [SING, "run", "--cleanenv"] + binds + [str(SIF_PATH)] + tool_args


# ---- Run the discovery pass for each subject ------------------------------------
for sub in SUBJECTS:
    rc = sh(heudiconv_cmd(sub, "convertall", "none"))   # convertall + no converter = catalog only
    if rc == 0:
        print(f"\n✅ sub-{sub}: discovery pass finished")
    else:
        print(f"\n❌ sub-{sub}: HeuDiConv failed (exit code {rc}). Read the messages above; "
              "the Troubleshooting table at the end of the notebook covers common causes.")


### Show the series table

When the discovery pass has finished, HeuDiConv has written one `dicominfo.tsv` per subject (and session) inside the hidden `.heudiconv/` folder. The cell below finds those files and displays the most useful columns, with one row per series. A comment next to each column name in the code explains what it means.

> 🔒 The full `dicominfo.tsv` also contains patient information (ID, age, sex). The cell only shows the columns needed to write the heuristic.

In [ ]:
import pandas as pd   # pandas reads tables (like .tsv files) and shows them nicely

# Show wide tables without cutting off columns or rows
pd.set_option("display.width", 250, "display.max_columns", 50, "display.max_rows", 200)

# Find every dicominfo table HeuDiConv wrote (one per subject/session) in the hidden .heudiconv folder.
# "**" means "look in all subfolders".
tsvs = sorted(BIDS_OUT.glob(".heudiconv/**/dicominfo*.tsv"))
print("Found:", *[str(t.relative_to(BIDS_OUT)) for t in tsvs], sep="\n  ")

# The columns that matter for writing the heuristic (the full file has many more).
# NOTE: the full dicominfo.tsv also contains patient info (ID, age, sex), another reason
# the .heudiconv folder must never be shared. We only display the columns below.
cols = ["series_id",           # number + name HeuDiConv uses to refer to each series
        "series_files",        # how many DICOM files in the series (compare with Step 2)
        "protocol_name",       # protocol name chosen at the scanner: usually the best handle
        "series_description",  # series name shown at the scanner (often the same as protocol_name)
        "dim1", "dim2", "dim3",  # image size: x, y, number of slices
        "dim4",                # number of volumes: 1 = a single 3D image (anat); >1 = 4D (BOLD, DWI)
        "TR", "TE",            # repetition time / echo time (in seconds)
        "is_derived",          # True = made by the scanner from other images (usually skip these)
        "image_type"]          # extra scanner labels, e.g. ORIGINAL / DERIVED, MOSAIC

if not tsvs:
    print("No dicominfo.tsv found. Did the discovery pass above finish without errors?")
for t in tsvs:
    info = pd.read_csv(t, sep="\t")                       # read the tab-separated table
    label = t.relative_to(BIDS_OUT / ".heudiconv").parent.parent   # e.g. 001/ses-01
    print(f"\n{label} — {len(info)} series")
    display(info[[c for c in cols if c in info.columns]])  # show only the columns listed above


### 👀 Read the table above

Each row is one **series** (one scan). Your job now is to decide which rows go into BIDS, and note what makes each one recognisable, so you can write rules for it in Step 4.

| Look at | Why |
|---|---|
| `protocol_name` / `series_description` | Usually the best handle (e.g. `T1_MPRAGE`, `rest_bold`). If your lab names protocols in a BIDS-like way (e.g. `anat-T1w`), matching is easy. |
| `dim4` | `1` = a single 3D image (anatomical, fieldmap); `> 1` = 4D (functional BOLD, diffusion). |
| `series_files` | A run with far fewer files than its repeats was probably stopped early. Decide whether to keep it. |
| `is_derived` / `image_type` | Scanner-made images (reformats, ADC/FA maps, MoCo series) are usually **skipped**. |
| Localizers / scouts | Positioning scans: always skip them. |

Write down the `protocol_name` of each scan you want to keep. Then adjust the rules in the heuristic in the next section.

## 4. Write the heuristic

The heuristic is a Python file with an `infotodict(seqinfo)` function. It receives one record per series and returns a dict mapping **BIDS filename templates → list of series IDs**.

The rules below cover common cases (T1w, T2w, FLAIR, resting/task BOLD, DWI, spin-echo fieldmaps). **Edit the keyword matches to fit the `protocol_name`s in your table.**

In [ ]:
HEURISTIC = r"""
# Auto-generated by the notebook. Edit the matching rules to fit your scanner protocols.

SES_DIR = "__SESDIR__"   # "{session}/" or ""
SES_TAG = "__SESTAG__"   # "_{session}" or ""
RUN = "__RUN__"          # "_run-{item:02d}" or ""

import re
ACQ_RE = re.compile(r"acq-([0-9a-z.]+)")   # picks up e.g. acq-0.8mm from the protocol name

# Task name -> keyword(s) found in protocol_name (lowercase)
TASKS = {
    "rest": ["rest", "rsfmri", "resting"],
    # "nback": ["nback", "n-back"],
}

# Fieldmaps only: to auto-fill \"IntendedFor\" in fieldmap sidecars, uncomment this block.
# POPULATE_INTENDED_FOR_OPTS = {
#     "matching_parameters": "ModalityAcquisitionLabel",
#     "criterion": "Closest",
# }

def create_key(template, outtype=("nii.gz",), annotation_classes=None):
    if not template:
        raise ValueError("Template must be a valid format string")
    return template, outtype, annotation_classes

def acq_tag(name):
    # BIDS labels must be alphanumeric, so "0.8mm" -> "0p8mm"
    m = ACQ_RE.search(name)
    if not m:
        return ""
    return "_acq-" + re.sub(r"[^0-9a-z]", "", m.group(1).replace(".", "p"))

def anat(suffix, name=""):
    return create_key("sub-{subject}/" + SES_DIR + "anat/sub-{subject}" + SES_TAG + acq_tag(name) + RUN + "_" + suffix)

def func(task, suffix="bold"):
    return create_key("sub-{subject}/" + SES_DIR + "func/sub-{subject}" + SES_TAG +
                      "_task-" + task + RUN + "_" + suffix)

def dwi_key():
    return create_key("sub-{subject}/" + SES_DIR + "dwi/sub-{subject}" + SES_TAG + RUN + "_dwi")

def fmap_epi(direction):
    return create_key("sub-{subject}/" + SES_DIR + "fmap/sub-{subject}" + SES_TAG +
                      "_dir-" + direction + RUN + "_epi")

def has(text, words):
    return any(w in text for w in words)

def infotodict(seqinfo):
    info = {}
    def add(key, s):
        info.setdefault(key, []).append(s.series_id)

    for s in seqinfo:
        p = (s.protocol_name or "").lower()
        d = (s.series_description or "").lower()
        name = p + " " + d
        itype = [str(x).upper() for x in (s.image_type or ())]

        # --- skip junk ---------------------------------------------------
        if s.is_derived or "DERIVED" in itype:
            continue
        if has(name, ["localizer", "scout", "survey", "aahead", "report", "phoenix"]):
            continue

        # --- fieldmaps (spin-echo EPI, AP/PA) -------------------------------
        if has(name, ["se_epi", "topup", "se-epi", "fmap"]) and s.dim4 <= 2:
            if has(name, ["_pa", "pa_", "-pa", "_p-a", "p>>a"]):
                add(fmap_epi("PA"), s); continue
            if has(name, ["_ap", "ap_", "-ap", "_a-p", "a>>p"]):
                add(fmap_epi("AP"), s); continue

        # --- DWI --------------------------------------------------------------
        if has(name, ["dwi", "dti", "diffusion"]) and s.dim4 > 1:
            add(dwi_key(), s); continue

        # --- functional -------------------------------------------------------
        if s.dim4 > 1:
            matched = False
            for task, kws in TASKS.items():
                if has(name, kws):
                    add(func(task), s); matched = True; break
            if matched:
                continue

        # --- anatomical -------------------------------------------------------
        if s.dim4 == 1:
            if has(name, ["flair"]):
                add(anat("FLAIR", name), s); continue
            if has(name, ["t2w", "t2_space", "t2_tse", "t2 space", "t2_"]):
                add(anat("T2w", name), s); continue
            if has(name, ["mprage", "t1w", "t1_", "spgr", "bravo", "t1 "]):
                add(anat("T1w", name), s); continue

        print("UNMATCHED series (not converted):", s.series_id, "|", s.protocol_name)

    return info
"""

if SESSION:
    HEURISTIC = HEURISTIC.replace("__SESDIR__", "{session}/").replace("__SESTAG__", "_{session}")
else:
    HEURISTIC = HEURISTIC.replace("__SESDIR__", "").replace("__SESTAG__", "")
HEURISTIC = HEURISTIC.replace("__RUN__", "_run-{item:02d}" if ADD_RUN else "")

HEURISTIC_PATH.parent.mkdir(parents=True, exist_ok=True)
HEURISTIC_PATH.write_text(HEURISTIC)
print("Wrote", HEURISTIC_PATH)

### Dry-run the heuristic against `dicominfo.tsv`

Before converting, check what the heuristic *would* do with the series from your discovery pass. This runs entirely in Python, so it's instant and safe.

In [ ]:
import ast, importlib.util
from collections import namedtuple

if tsvs:
    spec = importlib.util.spec_from_file_location("heuristic", HEURISTIC_PATH)
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)

    df = pd.read_csv(tsvs[0], sep="\t").fillna("")
    Row = namedtuple("Row", df.columns.tolist())
    seqinfo = []
    for r in df.to_dict("records"):
        it = r.get("image_type", "")
        if isinstance(it, str) and it.startswith("("):
            it = ast.literal_eval(it)
        r["image_type"] = it
        r["is_derived"] = str(r.get("is_derived", "")).lower() == "true"
        seqinfo.append(Row(**r))

    result = mod.infotodict(seqinfo)
    print("\nPlanned conversions:\n")
    for (tmpl, _, _), ids in result.items():
        print(f"  {tmpl}\n      <- series {ids}")
else:
    print("Run the discovery pass first.")

## 5. Convert

Runs HeuDiConv with your heuristic and `dcm2niix`, producing NIfTI files and JSON sidecars in BIDS layout.

With `SKIP_TOP_LEVEL = True` the command uses `-b notop`, so HeuDiConv **will not** write `dataset_description.json`, `participants.tsv`, `participants.json`, `README`, or `CHANGES`; your main dataset keeps its own. It still writes per-session `sub-*_ses-*_scans.tsv` files, which are normal BIDS and merge fine alongside the MEG data.

> The hidden `.heudiconv/` folder (which holds copies of DICOM headers) is always created in the output folder. That is why `BIDS_OUT` is a **staging** folder: convert there, then copy only the `sub-*` folders into the main dataset (Step 8) and leave `.heudiconv/` behind.

In [ ]:
for sub in SUBJECTS:
    sh(heudiconv_cmd(sub, str(HEURISTIC_PATH), "dcm2niix"))

## 6. Inspect the result

In [ ]:
def tree(root, max_depth=4):
    root = Path(root)
    for p in sorted(root.rglob("*")):
        rel = p.relative_to(root)
        if any(part.startswith(".") for part in rel.parts):   # hide .heudiconv, .git, etc.
            continue
        if len(rel.parts) > max_depth:
            continue
        print("  " * (len(rel.parts) - 1) + ("📁 " if p.is_dir() else "📄 ") + p.name)

tree(BIDS_OUT)

In [ ]:
import json

niftis = sorted(BIDS_OUT.glob("sub-*/**/*.nii.gz"))
print(f"{len(niftis)} NIfTI files\n")

# Sidecar sanity checks
for nii in niftis:
    side = nii.with_name(nii.name.replace(".nii.gz", ".json"))
    if not side.exists():
        print("MISSING sidecar:", side.name); continue
    meta = json.loads(side.read_text())
    problems = []
    if nii.name.endswith("_bold.nii.gz"):
        if "TaskName" not in meta:           problems.append("no TaskName")
        if "RepetitionTime" not in meta:     problems.append("no RepetitionTime")
    if "fmap" in nii.parts and "IntendedFor" not in meta:
        problems.append("no IntendedFor (fieldmap not linked to a scan)")
    print(("⚠️ " if problems else "✅ ") + nii.name, "|", ", ".join(problems))

## 7. Deface the anatomical images

T1w/T2w scans contain the participant's face, so they should be defaced before they go into a shared dataset. This step uses [**pydeface**](https://github.com/poldracklab/pydeface), which registers each image to a template with FSL's FLIRT and zeroes out the face region. Pydeface needs **FSL**, which is already inside the pydeface container you pulled at the top of the notebook.

How it works here:
* **Backup first.** The identifiable original of each image is copied to `BACKUP_DIR` before anything is changed, and the defaced image then replaces it **in place** in the staging folder (so the BIDS filenames stay the same). Keep `BACKUP_DIR` **outside** any dataset you'll share, and secure it like the raw DICOMs.
* **Each image is defaced independently.** Your T1w and T2w have different fields of view, so reusing one mask for both isn't safe.
* **A log** (`.deface_log.tsv`, hidden in the staging folder) records what's been done, so re-running the cell won't deface things twice.
* **Always look at the QC images below.** Defacing can occasionally clip brain, especially on T2w. If one fails, try `--cost corratio` or `--nocleanup` flags (see `pydeface --help`), or a different tool such as `mri_reface` or `mri_deface`.

In [ ]:
# ---- Defacing config ----------------------------------------------------------
DEFACE_SUFFIXES  = ["T1w", "T2w"]            # add "FLAIR" etc. if needed

# Only if auto-detection fails: full path to the pydeface executable INSIDE the image
# (the diagnostics printed in Step 7 will show where it is), e.g. "/usr/local/miniconda/bin/pydeface"
PYDEFACE_EXECUTABLE = None
BACKUP_DIR       = BIDS_OUT.parent / "unfaced_backup"   # KEEP OUTSIDE ANY SHARED DATASET
REDEFACE         = False                     # True -> deface again even if the log says it's done

# ---- Environment check -----------------------------------------------------------
print("pydeface image exists:", PYDEFACE_SIF.exists(), "->", PYDEFACE_SIF)

In [ ]:
LOG = BIDS_OUT / ".deface_log.tsv"
done = {l.split("\t")[0] for l in LOG.read_text().splitlines()[1:]} if LOG.exists() else set()

def _container_cmd(mode, bind_dir, tail):
    """Build the apptainer command for one of three ways of launching pydeface."""
    b = ["-B", f"{Path(bind_dir).resolve()}:/work"]
    if mode == "exec_name":          # pydeface is on the image's PATH
        return [SING, "exec", "--cleanenv"] + b + [str(PYDEFACE_SIF), "pydeface"] + tail
    if mode == "exec_path":          # explicit path inside the image
        return [SING, "exec", "--cleanenv"] + b + [str(PYDEFACE_SIF), PYDEFACE_EXECUTABLE] + tail
    if mode == "run":                # the image's entrypoint IS pydeface
        return [SING, "run", "--cleanenv"] + b + [str(PYDEFACE_SIF)] + tail
    raise ValueError(mode)

def _probe(mode):
    try:
        r = subprocess.run(_container_cmd(mode, ".", ["--help"]), capture_output=True, text=True, timeout=180)
    except Exception:
        return False
    return r.returncode == 0 and "usage" in (r.stdout + r.stderr).lower()

def detect_pydeface_mode():
    if PYDEFACE_EXECUTABLE:
        return "exec_path"
    for mode in ("exec_name", "run"):
        if _probe(mode):
            return mode
    print("Could not launch pydeface from this image. Diagnostics:\n")
    sh([SING, "inspect", "--runscript", str(PYDEFACE_SIF)])
    sh([SING, "exec", "--cleanenv", str(PYDEFACE_SIF), "sh", "-c",
        "echo PATH=$PATH; find / -xdev -name 'pydeface*' 2>/dev/null | head -20"])
    raise RuntimeError("Set PYDEFACE_EXECUTABLE (path inside the image, from the output above) in the config cell, "
                       "or check that PYDEFACE_SIF really is a pydeface image.")

PYDEFACE_MODE = detect_pydeface_mode()
print("pydeface launch mode:", PYDEFACE_MODE, "\n")

def pydeface_cmd(infile, outfile):
    tail = [f"/work/{infile.name}", "--outfile", f"/work/{outfile.name}", "--force"]
    return _container_cmd(PYDEFACE_MODE, infile.parent, tail)

targets = []
for sub in SUBJECTS:
    for suf in DEFACE_SUFFIXES:
        targets += sorted((BIDS_OUT / f"sub-{sub}").rglob(f"anat/*_{suf}.nii.gz"))
print(f"{len(targets)} image(s) to consider\n")

if not LOG.exists():
    LOG.write_text("file\tdefaced_at\ttool\n")

for f in targets:
    rel = f.relative_to(BIDS_OUT)
    if str(rel) in done and not REDEFACE:
        print("already defaced, skipping:", rel); continue

    backup = BACKUP_DIR / rel
    backup.parent.mkdir(parents=True, exist_ok=True)
    if not backup.exists():                 # never overwrite the true original
        shutil.copy2(f, backup)

    tmp = f.with_name(f.name.replace(".nii.gz", "_defacetmp.nii.gz"))
    rc = sh(pydeface_cmd(f, tmp))
    if rc != 0 or not tmp.exists():
        print(f"\nFAILED: {rel}  (original left untouched)\n")
        tmp.unlink(missing_ok=True)
        continue
    os.replace(tmp, f)                      # defaced image takes the BIDS filename
    with LOG.open("a") as fh:
        fh.write(f"{rel}\t{datetime.datetime.now().isoformat(timespec='seconds')}\tpydeface ({PYDEFACE_SIF.name})\n")
    print(f"\nDEFACED: {rel}\n")

In [ ]:
# Visual QC: original vs defaced (mid-sagittal + an axial slice through the face)
try:
    import nibabel as nib, numpy as np, matplotlib.pyplot as plt
except ImportError:
    print("nibabel isn't installed. See 'Install the Python packages' at the top, then restart the kernel and re-run this cell.")
else:
    pairs = [(BACKUP_DIR / f.relative_to(BIDS_OUT), f) for f in targets if (BACKUP_DIR / f.relative_to(BIDS_OUT)).exists()]
    if not pairs:
        print("Nothing to show yet.")
    else:
        fig, axes = plt.subplots(len(pairs), 4, figsize=(16, 4 * len(pairs)), squeeze=False)
        for row, (orig, new) in enumerate(pairs):
            for col, (path, label) in enumerate([(orig, "original"), (new, "defaced")]):
                data = nib.as_closest_canonical(nib.load(path)).get_fdata()
                vmax = np.percentile(data, 99.5)
                sag = data[data.shape[0] // 2, :, :].T
                axi = data[:, :, int(data.shape[2] * 0.35)].T
                for k, (sl, view) in enumerate([(sag, "sagittal"), (axi, "axial")]):
                    ax = axes[row, col * 2 + k]
                    ax.imshow(sl, cmap="gray", origin="lower", vmax=vmax)
                    ax.set_title(f"{label} - {view}\n{new.name[:42]}", fontsize=8)
                    ax.axis("off")
        plt.tight_layout(); plt.show()

## 8. Add the converted data to your main dataset (optional)

Copies only the `sub-*` folders from the staging folder into `MAIN_DATASET`, leaving `.heudiconv/` and anything else behind. It **never overwrites** existing files (`--ignore-existing`), so it won't touch the subject's `meg/` data. Leave `DRY_RUN = True` first to see what would be copied.

In [ ]:
DRY_RUN = True      # flip to False to actually copy

if MAIN_DATASET is None:
    print("MAIN_DATASET is None -> skipping. Set it in the config cell to use this step.")
else:
    MAIN_DATASET = Path(MAIN_DATASET)

    # Safety: refuse to copy identifiable anatomicals that haven't been defaced
    _log = BIDS_OUT / ".deface_log.tsv"
    _done = {l.split("\t")[0] for l in _log.read_text().splitlines()[1:]} if _log.exists() else set()
    _undefaced = [str(p.relative_to(BIDS_OUT)) for sub in SUBJECTS
                  for suf in ("T1w", "T2w", "FLAIR")
                  for p in (BIDS_OUT / f"sub-{sub}").rglob(f"anat/*_{suf}.nii.gz")
                  if str(p.relative_to(BIDS_OUT)) not in _done]
    if _undefaced and not DRY_RUN:
        raise RuntimeError("Not defaced yet (run Step 7 first):\n  " + "\n  ".join(_undefaced))
    elif _undefaced:
        print("WARNING - these have not been defaced and a real copy would be blocked:\n  " + "\n  ".join(_undefaced), "\n")

    for sub in SUBJECTS:
        src = BIDS_OUT / f"sub-{sub}"
        if not src.exists():
            print("nothing to copy for", src); continue
        dest = MAIN_DATASET / f"sub-{sub}"
        dest.mkdir(parents=True, exist_ok=True)
        cmd = ["rsync", "-av", "--ignore-existing"] + (["--dry-run"] if DRY_RUN else []) + [f"{src}/", f"{dest}/"]
        sh(cmd)
    if DRY_RUN:
        print("\nDry run only. Set DRY_RUN = False to copy.")

## 9. Validate with the BIDS validator

A standalone staging folder has no `dataset_description.json`, so the validator would report errors that don't matter. When `SKIP_TOP_LEVEL` is on, this step therefore validates **`MAIN_DATASET`** (after Step 8) rather than the staging folder.

In [ ]:
# Uses the bids-validator container (VALIDATOR_SIF) pulled at the top of the notebook.

TARGET = MAIN_DATASET if (SKIP_TOP_LEVEL and MAIN_DATASET) else BIDS_OUT
if SKIP_TOP_LEVEL and not MAIN_DATASET:
    print("Skipping: staging folder has no top-level files by design. Set MAIN_DATASET, copy (Step 8), then validate it.")
elif SING and VALIDATOR_SIF.exists():
    sh([SING, "run", "--cleanenv", "-B", f"{Path(TARGET).resolve()}:/data:ro", VALIDATOR_SIF, "/data"])
else:
    print("No validator found. Check VALIDATOR_SIF and pull the image (see top of notebook),\n"
          "or drop the folder into the browser validator (files stay local): https://bids-standard.github.io/bids-validator/")

## Troubleshooting

| Symptom | Likely fix |
|---|---|
| `0 files match` | Check `DICOM_TEMPLATE` and that the subject folder name matches `SUBJECTS` exactly. |
| `UNMATCHED series` | Add its `protocol_name` keywords to the heuristic rules. |
| `dataset_description.json` etc. appeared anyway | `SKIP_TOP_LEVEL` was `False`, or you re-ran into a folder that already had them. Delete them and re-run with it `True`. |
| pydeface fails / `flirt: command not found` | pydeface needs FSL, which is inside the pydeface container. Check that `PYDEFACE_SIF` points to the image you pulled. |
| Defacing clipped brain tissue | Restore from `BACKUP_DIR`, set `REDEFACE = True`, and retry with other pydeface options (`--cost corratio`) or another tool (`mri_reface`). |
| Output files named `run-01`, `run-02` for one scan | The series was split (e.g., repeated/incomplete run). Check `dicominfo.tsv` for duplicates; filter on `dim3`/`dim4`. |
| `Conversion directory already exists` / nothing re-converts | Set `OVERWRITE = True` (or delete `.heudiconv/<sub>`). |
| Multiple studies/sessions mixed in one folder | Add `--grouping accession_number` (or `study_uid`) to the command. |
| Subject ID rejected | BIDS labels are alphanumeric only. Use `--anon-cmd` or rename the folder. |
| Fieldmap missing `IntendedFor` | Check fieldmap series were matched and that `POPULATE_INTENDED_FOR_OPTS` is in your heuristic. |
| `FATAL: could not open image` | Check `SIF_PATH` (the config cell's path check should catch this). |
| `Apptainer/Singularity not found` | Make sure Jupyter is running on the cluster, not on your laptop. |
| Container can't see a folder (e.g. `/scratch`) | Add it to `EXTRA_BINDS`, and make sure `DICOM_ROOT` / `BIDS_OUT` use real paths (not symlinks that point outside bound folders). |
| `Read-only file system` writing to `/home` | Singularity mounts `$HOME`; keep `BIDS_OUT` on project/scratch storage and bound via `EXTRA_BINDS` if needed. |

**Next steps:** add the new subjects to your main dataset's `participants.tsv` (HeuDiConv no longer touches it), and store `BACKUP_DIR` (the unfaced originals) with the same access controls as your raw DICOMs.